![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# FrankenMSA-Colab
This notebook launches the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) **in Google Colab** to provide a GUI for manipulating Multiple Sequence Alignments (MSAs).


Tip: use “Runtime” → “Run all” (or `Ctrl + F9`) to execute all cells.

In [1]:
FRANKEN_GIT_BRANCH = "feature/colab-runner-v2" # "main" # change as needed (default: main)
FRANKEN_GIT_URL = "https://github.com/ibmm-unibe-ch/FrankenMSA.git"

In [1]:
#@title Install Prerequisites

!pip install termcolor gitpython ipywidgets ipython

import git, os, sys, importlib
from pathlib import Path
from termcolor import colored

def warn(msg):
    print(colored("[WARNING] ", "yellow") + msg, file=sys.stderr)

def info(msg):
    print(colored("[INFO] ", "cyan") + msg)

In [2]:
#@title Prepare Rendering and Sharing options

import ipywidgets as widgets
from IPython.display import display

_render_dropdown = widgets.Dropdown(
    options=[
        ("External browser tab", "external"),
    ],
    value="external",
    description="Render:",
)
_ngrok_checkbox = widgets.Checkbox(
    value=False,
    description="Create ngrok share link",
)
_port_input = widgets.IntText(
    value=8050,
    description="Port:",
    min=1024,
    max=65535,
)
_status = widgets.Output()

use_ngrok = lambda : _ngrok_checkbox.value

def _enforce_render_on_ngrok(change):
    if change["name"] != "value":
        return
    with _status:
        _status.clear_output()
        if change["new"]:
            if _render_dropdown.value != "external":
                _render_dropdown.value = "external"
            info("ngrok sharing forces external rendering.")
        else:
            info("ngrok sharing disabled; inline mode available.")

def _enforce_ngrok_on_render(change):
    if change["name"] != "value":
        return
    if use_ngrok() and change["new"] != "external":
        with _status:
            _status.clear_output()
            info("ngrok sharing forces external rendering.")
        _render_dropdown.value = "external"

_ngrok_checkbox.observe(_enforce_render_on_ngrok, names="value")
_render_dropdown.observe(_enforce_ngrok_on_render, names="value")
display(widgets.VBox([_render_dropdown, _ngrok_checkbox, _port_input, _status]))

In [15]:
#@title Install FrankenMSA

FORCE_REINSTALL = False #@param {type:"boolean"}

if Path("frankenmsa").exists() and not FORCE_REINSTALL:
    warn("FrankenMSA directory already exists; skipping clone.")
else:
    os.system("rm -rf FrankenMSA frankenmsa app; rm -f *.py")
    info(f"Cloning FrankenMSA from {FRANKEN_GIT_URL} (branch: {FRANKEN_GIT_BRANCH})...")
    git.Repo.clone_from(FRANKEN_GIT_URL, "FrankenMSA", branch=FRANKEN_GIT_BRANCH)
    info("FrankenMSA cloned.")

    os.system("mv FrankenMSA/app .; mv FrankenMSA/frankenmsa .; mv FrankenMSA/setup.py .; rm -rf FrankenMSA; pip install -e .")
    info("FrankenMSA installed.")

if use_ngrok():
      try:
          import pyngrok
      except:
          info("Installing pyngrok...")
          os.system("pip install pyngrok; pip install getpass")
          info("pyngrok installed.")
      try:
        from pyngrok import ngrok

      except:
        raise ImportError("Pyngrok could not be installed")

# kill any existing instances
!pkill -f "app/app.py" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
!pkill -f "ngrok" 2>/dev/null || true

[WARNING] FrankenMSA directory already exists; skipping clone.


^C
^C
^C


In [19]:
#@title Launch FrankenMSA App (without ngrok sharing)
if not use_ngrok():
    from app.app import launch
    launch(
        render_mode=_render_dropdown.value,
        port=_port_input.value,
    )

Dash is starting on http://0.0.0.0:8050
Dash is running on http://0.0.0.0:8050/



INFO:dash.dash:Dash is running on http://0.0.0.0:8050/



 * Serving Flask app 'app.app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8050
 * Running on http://172.28.0.12:8050
INFO:werkzeug:Press CTRL+C to quit


In [17]:
#@title Launch FrankenMSA App (with ngrok sharing)
if use_ngrok():

    from pyngrok import ngrok, conf
    import getpass, re

    token = os.environ.get("NGROK_AUTH_TOKEN", "").strip()
    if not token:
        warn("No ngrok auth token found in NGROK_AUTH_TOKEN env variable.")
        warn("You can sign up for a free ngrok account at https://ngrok.com/")
        warn("To avoid entering the token every time, the token is set it in the NGROK_AUTH_TOKEN environment variable after entering.")

        token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
        os.environ["NGROK_AUTH_TOKEN"] = token

        info("ngrok auth token set as environment variable.")
    conf.get_default().auth_token = token

    public_url = None
    PORT = _port_input.value
    try:
        for t in ngrok.get_tunnels():
            addr = (t.config or {}).get("addr", "")
            if addr.endswith(f":{PORT}"):
                public_url = t.public_url
                print("♻️ Reusing existing tunnel:", public_url)
                break

        if not public_url:
            tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
            public_url = tun.public_url
            print("✅ Created new tunnel:", public_url)

    except Exception as e:
        msg = str(e)
        m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
        if m:
            public_url = m.group(0)
            warn("♻️ Using tunnel from error message:", public_url)
        else:
            raise e

    # fix initial callbacks to use prevent_initial_call=True
    # (not sure why this patching is needed - should this not be a part of the codebase?)
    patch_total = 0
    for p in Path("app").rglob("*.py"):
        s = p.read_text(encoding="utf-8", errors="ignore")
        s2, n1 = re.subn(r",\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}", ", prevent_initial_call=True", s)
        s3, n2 = re.subn(
            r"clientside_callback\((.*?)\s*,\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}\s*\)",
            r"clientside_callback(\1, prevent_initial_call=True)",
            s2, flags=re.DOTALL
        )
        if n1 or n2:
            p.write_text(s3, encoding="utf-8")
            patch_total += n1 + n2
    f = warn if patch_total else info
    f(f"🩹 Patched {patch_total} place(s)")


    import subprocess, time

    # setup environment variables for subprocess app launch
    env = os.environ.copy()
    env["PORT"], env["HOST"] = str(PORT), os.environ.get("HOST", "0.0.0.0")
    env["PYTHONPATH"] = "/content:" + env.get("PYTHONPATH", "")
    env["FRANKEN_COLAB"] = "1"
    env["IN_COLAB"] = "1"
    env["FRANKEN_RENDER_MODE"] = _render_dropdown.value
    env["FRANKEN_USE_NGROK"] = "1" if _ngrok_checkbox.value else "0"
    if public_url:
        env["COLAB_TUNNEL_URL"] = public_url
    else:
        env.pop("COLAB_TUNNEL_URL", None)

    proc = subprocess.Popen(
        [sys.executable, "app/app.py"],
        cwd=str(Path.cwd().resolve()),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    start = time.time()
    lines = []
    while time.time() - start < 25:
        ln = proc.stdout.readline()
        if ln:
            lines.append(ln.rstrip())
            if "Running on" in ln or "Dash is running" in ln:
                break
        else:
            time.sleep(0.2)

    display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "127.0.0.1"
    open_target = public_url if public_url else f"http://{display_host}:{PORT}"

    print("\n---- recent logs ----")
    print("\n".join(lines[-20:]))
    print("---------------------")
    info(f"🌐 Open: {open_target}")
    # print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
    print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
    while True:
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.2)
            continue
        print(line, end="")

♻️ Reusing existing tunnel: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev
[INFO] 🩹 Patched 0 place(s)

---- recent logs ----
[UPLOAD_DIR] using: /content/ProteinMPNN/uploads
Dash is starting on http://0.0.0.0:8050
🌐 Public tunnel: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev
Dash is running on http://0.0.0.0:8050/
---------------------
[INFO] 🌐 Open: https://gracelynn-uncatastrophic-organizingly.ngrok-free.dev
📡 Tailing FrankenMSA app logs (Ctrl+C to stop):

 * Serving Flask app 'app'
 * Debug mode: off
Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


KeyboardInterrupt: 

In [ ]:
# # === Cell 1: Hard reset + fresh clone of the RIGHT branch ===
# import os, sys, shutil, subprocess, time, json, textwrap, pathlib

# # --- Clean any old tunnels/processes (best-effort) ---



# # --- Fresh clone the correct branch ---
# shutil.rmtree("/content/FrankenMSA", ignore_errors=True)
# !git clone -q --single-branch -b feature/colab-runner https://github.com/ibmm-unibe-ch/FrankenMSA.git /content/FrankenMSA

# # show branch & last commit for sanity
# !git -C /content/FrankenMSA rev-parse --abbrev-ref HEAD
# !git -C /content/FrankenMSA log -1 --pretty=oneline

# # --- Base deps for the web app / ngrok  ---
# try:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
#                            "dash==2.16.1", "dash-bootstrap-components==1.5.0",
#                            "plotly==5.24.1", "dash-bio==1.0.2", "jupyter-dash==0.4.2",
#                            "pyngrok"])
#     print("✅ base deps installed")
# except Exception as e:
#     print("⚠️ deps warning:", e)

# # --- Inject build banner into inversefold.py ---
# try:
#     commit = subprocess.check_output(
#         ["git","-C","/content/FrankenMSA","rev-parse","--short","HEAD"],
#         text=True
#     ).strip()
#     p = pathlib.Path("/content/FrankenMSA/app/pages/inversefold.py")
#     s = p.read_text(encoding="utf-8")
#     if "INVERSEFOLD_BUILD_ID" not in s:
#         s = s.replace(
#             "def layout():",
#             f"INVERSEFOLD_BUILD_ID = 'IF-build:{commit}'\n\ndef layout():"
#         ).replace(
#             'html.H1("Inverse Fold with ProteinMPNN")',
#             'html.H1("Inverse Fold with ProteinMPNN"), html.Small(INVERSEFOLD_BUILD_ID, style={"marginLeft":"8px","opacity":0.6})'
#         )
#         p.write_text(s, encoding="utf-8")
#         print("🧩 injected build banner:", commit)
#     else:
#         print("ℹ️ build banner already present")
# except Exception as e:
#     print("⚠️ build banner injection skipped:", e)

# # --- Export /content/colab_bridge.py  ---
# bridge_code = r'''
# import sys

# if "/content/FrankenMSA/app" not in sys.path:
#     sys.path.append("/content/FrankenMSA/app")

# import proteinmpnn_runner as pmr

# def parse_params(qs_or_url: str):
#     """
#     Normalize FrankenMSA URL/query-string to parameter dict.
#     (Web app expects this symbol name.)
#     """
#     return pmr.parse_param_string(qs_or_url)

# def run_proteinmpnn(
#     sampling_temp=1.0, num_seqs=128, pdb_code="", design_csv="", fixed_csv="",
#     homomer=True, model_name="v_48_020", use_soluble_model=False, ca_only=False,
#     clean_workspace=True, allow_upload=False, auto_download=False,
#     pdb_path=""  # optional: if later you want to pass an uploaded local PDB file path
# ):

#     return pmr.run_proteinmpnn(
#         sampling_temp=sampling_temp,
#         num_seqs=num_seqs,
#         pdb_code=pdb_code,
#         design_csv=design_csv,
#         fixed_csv=fixed_csv,
#         homomer=homomer,
#         model_name=model_name,
#         use_soluble_model=use_soluble_model,
#         ca_only=ca_only,
#         allow_upload=allow_upload,
#         clean_workspace=clean_workspace,
#         auto_download=auto_download,
#         pdb_path=pdb_path,
#     )
# '''
# with open("/content/colab_bridge.py", "w") as f:
#     f.write(bridge_code)


# if "/content" not in sys.path:
#     sys.path.append("/content")
# !mkdir -p /content/ProteinMPNN/uploads

# # Smoke test
# import importlib
# colab_bridge = importlib.import_module("colab_bridge")
# print("✅ colab_bridge exported:", [n for n in dir(colab_bridge) if not n.startswith("_")])

# print("✅ Cell 1 done. Go to Cell 2.")

In [ ]:
# # === Cell 2: Start app with selected rendering ===
# import os, signal, subprocess, time

# !pkill -f "app/app.py" 2>/dev/null || true
# !pkill -f "gunicorn" 2>/dev/null || true
# !pkill -f "ngrok" 2>/dev/null || true

# time.sleep(1)
# print("🧹 Cleaned up old FrankenMSA + ngrok processes.")

# import sys, re
# from pathlib import Path

# ROOT = "/content/FrankenMSA"
# assert os.path.isdir(ROOT), "FrankenMSA repo not found. Run Cell 1 first."

# render_mode = str(getattr(_render_dropdown, "value", "external") or "external").lower()
# if render_mode not in {"inline", "external"}:
#     render_mode = "external"

# use_ngrok = bool(getattr(_ngrok_checkbox, "value", True))
# if use_ngrok and render_mode != "external":
#     try:
#         if getattr(_render_dropdown, "value", None) != "external":
#             _render_dropdown.value = "external"
#     except Exception:
#         pass
#     render_mode = "external"
#     print("🔁 Forcing external rendering because ngrok sharing is enabled.")

# PORT = int(getattr(_port_input, "value", 8050) or 8050)
# if PORT < 1024 or PORT > 65535:
#     PORT = 8050
#     print(f"⚠️ Invalid port; resetting to 8050.")

# print(f"🎨 Rendering mode: {render_mode}")
# print(f"🌐 ngrok sharing: {'enabled' if use_ngrok else 'disabled'}")
# print(f"🔌 Port: {PORT}")

# public_url = None

# if use_ngrok:
#     from pyngrok import ngrok, conf
#     import getpass

#     token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
#     conf.get_default().auth_token = token

#     try:
#         for t in ngrok.get_tunnels():
#             addr = (t.config or {}).get("addr", "")
#             if addr.endswith(f":{PORT}"):
#                 public_url = t.public_url
#                 print("♻️ Reusing existing tunnel:", public_url)
#                 break

#         if not public_url:
#             tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
#             public_url = tun.public_url
#             print("✅ Created new tunnel:", public_url)
#     except Exception as e:
#         msg = str(e)
#         m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
#         if m:
#             public_url = m.group(0)
#             print("♻️ Using tunnel from error message:", public_url)
#         else:
#             raise
# else:
#     print("🚪 Skipping ngrok tunnel setup; the app will remain local to this notebook.")

# # --- Colab flags (make them visible to the app) ---
# os.environ["FRANKEN_COLAB"] = "1"
# os.environ["IN_COLAB"] = "1"
# os.environ["FRANKEN_RENDER_MODE"] = render_mode
# os.environ["PORT"] = str(PORT)
# os.environ.setdefault("HOST", "0.0.0.0")
# if public_url:
#     os.environ["COLAB_TUNNEL_URL"] = public_url
#     print("ENV COLAB_TUNNEL_URL:", public_url)
# else:
#     os.environ.pop("COLAB_TUNNEL_URL", None)

# patch_total = 0
# for p in Path(f"{ROOT}/app").rglob("*.py"):
#     s = p.read_text(encoding="utf-8", errors="ignore")
#     s2, n1 = re.subn(r",\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}", ", prevent_initial_call=True", s)
#     s3, n2 = re.subn(
#         r"clientside_callback\((.*?)\s*,\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}\s*\)",
#         r"clientside_callback(\1, prevent_initial_call=True)",
#         s2, flags=re.DOTALL
#     )
#     if n1 or n2:
#         p.write_text(s3, encoding="utf-8")
#         patch_total += n1 + n2
# print(f"🩹 Patched {patch_total} place(s)")

# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ROOT])

# if render_mode == "inline" and not use_ngrok:
#     import importlib

#     importlib.invalidate_caches()
#     print("✅ Launching FrankenMSA inline. The dashboard should appear above this cell.")
#     app_module = importlib.import_module("app.app")
#     app_module = importlib.reload(app_module)
#     app_module.launch()
# else:
#     env = os.environ.copy()
#     env["PORT"], env["HOST"] = str(PORT), os.environ.get("HOST", "0.0.0.0")
#     env["PYTHONPATH"] = "/content:" + env.get("PYTHONPATH", "")
#     env["FRANKEN_COLAB"] = "1"
#     env["IN_COLAB"] = "1"
#     env["FRANKEN_RENDER_MODE"] = render_mode
#     env["FRANKEN_USE_NGROK"] = "1" if use_ngrok else "0"
#     if public_url:
#         env["COLAB_TUNNEL_URL"] = public_url
#     else:
#         env.pop("COLAB_TUNNEL_URL", None)

#     proc = subprocess.Popen(
#         [sys.executable, "app/app.py"],
#         cwd=ROOT,
#         env=env,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.STDOUT,
#         text=True,
#         bufsize=1
#     )

#     start = time.time()
#     lines = []
#     while time.time() - start < 25:
#         ln = proc.stdout.readline()
#         if ln:
#             lines.append(ln.rstrip())
#             if "Running on" in ln or "Dash is running" in ln:
#                 break
#         else:
#             time.sleep(0.2)

#     display_host = env["HOST"] if env["HOST"] not in {"0.0.0.0", "::"} else "127.0.0.1"
#     open_target = public_url if public_url else f"http://{display_host}:{PORT}"

#     print("\n---- recent logs ----")
#     print("\n".join(lines[-20:]))
#     print("---------------------")
#     print("🌐 Open:", open_target)
#     print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")
#     print("📡 Tailing FrankenMSA app logs (Ctrl+C to stop):")
#     while True:
#         line = proc.stdout.readline()
#         if not line:
#             time.sleep(0.2)
#             continue
#         print(line, end="")